# 02_01：前 300 名 PERMNO 人工審核

- 來源為新版 02 的 `comment_ticker_long.parquet`，必須包含依留言日期對應成功的 `permno`。
- 每個成功 comment–ticker–PERMNO 列計一次 `times`；不改用 thread 數。同一 PERMNO 的不同 ticker 合併計數，不跨留言去重；若同則留言提及同 PERMNO 的不同 ticker，仍按來源列分別計次。
- 依 `times` 降冪，同分按數值 PERMNO 升冪，取前 300 名（不足 300 時取全部）。
- CSV 欄位固定為 `permno,ticker,times,del`；ticker 以 `|` 串接該 PERMNO 在輸入中出現的所有 ticker，按字母排序。
- 首次建立 `del=1`（刪除），人工可改為 `0`（保留）。既有審核表只檢查、顯示，不覆寫或重設人工 del。
- 新輸出：`Data/process/02_01_permno_frequency_review/<02 run>/top300_permno_review.csv`；保留舊 ticker 審核表。
- 本階段不執行刪除。既有 02_02 尚使用舊 ticker 審核規則，不能直接套用此 PERMNO 審核表。


In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib, json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Data/process').is_dir())
MATCH_RUN = None  # 可指定含 permno 的成功 02 run 資料夾。
if MATCH_RUN is None:
    value = json.loads((ROOT/'Data/process/pipeline_runs.json').read_text(encoding='utf-8')).get('matching_manifest')
    if not value: raise FileNotFoundError('請先成功執行新版 02。')
    reference = Path(value)
    if not reference.is_absolute(): reference = ROOT/reference
    MATCH_RUN = reference.parent
else:
    MATCH_RUN = Path(MATCH_RUN)
    if not MATCH_RUN.is_absolute(): MATCH_RUN = ROOT/MATCH_RUN
MATCH_RUN = MATCH_RUN.resolve()
SOURCE = MATCH_RUN/'comment_ticker_long.parquet'
OUT = ROOT/'Data/process/02_01_permno_frequency_review'/MATCH_RUN.name
REVIEW_CSV = OUT/'top300_permno_review.csv'
META = OUT/'run_summary.json'
if not SOURCE.is_file(): raise FileNotFoundError('請先執行 02 的 long-format 輸出。')
assert json.loads((MATCH_RUN/'run_summary.json').read_text(encoding='utf-8'))['status'] == 'complete'
if not {'permno', 'ticker'} <= set(pq.read_schema(SOURCE).names):
    raise ValueError('02 輸入缺少 permno；請重新執行新版 02，不會以 ticker 猜測 PERMNO。')
print('來源：', SOURCE)
print('審核檔：', REVIEW_CSV)


來源： C:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-10-13_05-01-55_UTC\comment_ticker_long.parquet
審核檔： c:\Users\eagle\RedditStock\Data\process\02_01_permno_frequency_review\2026-10-13_05-01-55_UTC\top300_permno_review.csv


## 統計與人工審核保護
保留既有 CSV 的所有人工內容。PERMNO 必須唯一且為正整數，times 為正整數，del 僅接受 0 或 1。來源內容改變時停止。


In [3]:
def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b''): digest.update(block)
    return digest.hexdigest()

def validate_review(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    assert frame.columns.tolist() == ['permno','ticker','times','del'], '欄位不符；原檔不改寫。'
    assert frame.permno.str.fullmatch(r'[1-9][0-9]*').all()
    assert frame.permno.is_unique and frame.ticker.str.strip().ne('').all()
    assert frame.times.str.fullmatch(r'[1-9][0-9]*').all()
    assert frame['del'].isin(['0','1']).all(), 'del 只能填 0 或 1。'
    assert len(frame) <= 300
    return frame

def ranked_review(frequency, tickers, top_n=300):
    ranked = sorted(frequency.items(), key=lambda item: (-item[1], item[0]))[:top_n]
    return pd.DataFrame([{'permno': permno, 'ticker': '|'.join(sorted(tickers[permno])),
                          'times': count, 'del': 1} for permno, count in ranked],
                        columns=['permno','ticker','times','del'])

# 合併同 PERMNO 的 ticker、數值同分排序、只取前300名。
test_review = ranked_review({20: 3, 3: 3, 1: 5}, {20: {'Z'}, 3: {'B','A'}, 1: {'X'}}, 2)
assert test_review.permno.tolist() == [1, 3]
assert test_review.ticker.tolist() == ['X', 'A|B']
assert test_review['del'].tolist() == [1, 1]

if REVIEW_CSV.exists():
    before_hash = file_hash(REVIEW_CSV)
    review = validate_review(REVIEW_CSV)
    if not META.is_file(): raise ValueError('既有審核表缺少來源摘要；原檔保留。')
    metadata = json.loads(META.read_text(encoding='utf-8'))
    assert Path(metadata['source']).resolve() == SOURCE
    assert metadata['source_sha256'] == file_hash(SOURCE), '來源已變更；不改寫人工審核表。'
    assert file_hash(REVIEW_CSV) == before_hash
    print('既有審核表保留所有人工修改。')
else:
    source_stat = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
    source_hash = file_hash(SOURCE)
    frequency, tickers = Counter(), defaultdict(set)
    total = 0
    for batch_no, batch in enumerate(pq.ParquetFile(SOURCE).iter_batches(batch_size=100_000, columns=['permno','ticker']), 1):
        frame = batch.to_pandas()
        ids = pd.to_numeric(frame.permno, errors='coerce')
        assert ids.notna().all() and ids.gt(0).all() and ids.mod(1).eq(0).all(), '無效 PERMNO；停止。'
        assert frame.ticker.notna().all() and frame.ticker.str.strip().ne('').all(), '空 ticker；停止。'
        frame['permno'] = ids.astype('int64')
        for permno, count in frame.permno.value_counts().items(): frequency[int(permno)] += int(count)
        for permno, ticker in frame[['permno','ticker']].drop_duplicates().itertuples(index=False, name=None):
            tickers[int(permno)].add(ticker)
        total += len(frame)
        if batch_no % 25 == 0: print({'batch':batch_no, 'rows':total, 'unique_permnos':len(frequency)})
    assert total == pq.read_metadata(SOURCE).num_rows == sum(frequency.values())
    review = ranked_review(frequency, tickers)
    assert len(review) == min(300, len(frequency))
    assert source_stat == (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
    OUT.mkdir(parents=True, exist_ok=True)
    with REVIEW_CSV.open('x', encoding='utf-8-sig', newline='') as stream: review.to_csv(stream, index=False)
    validate_review(REVIEW_CSV)
    metadata = {'status':'complete', 'source':str(SOURCE), 'source_sha256':source_hash,
                'total_mention_rows':total, 'unique_permnos':len(frequency), 'review_rows':len(review),
                'columns':['permno','ticker','times','del'], 'ranking':'times descending, numeric permno ascending',
                'count_unit':'one successful input comment-ticker-permno row', 'ticker_separator':'|',
                'initial_del':1, 'del_semantics':{'1':'刪除','0':'保留'}, 'deletions_applied':False}
    with META.open('x', encoding='utf-8') as stream: json.dump(metadata, stream, ensure_ascii=False, indent=2)
    print({'mention_rows':total, 'unique_permnos':len(frequency), 'review_rows':len(review)})
with pd.option_context('display.max_rows', 300): display(review)
print('請手動修改 del；本 notebook 不套用刪除。')


{'batch': 25, 'rows': 2500000, 'unique_permnos': 3065}
{'batch': 50, 'rows': 5000000, 'unique_permnos': 3431}
{'batch': 75, 'rows': 7500000, 'unique_permnos': 3574}
{'batch': 100, 'rows': 10000000, 'unique_permnos': 3765}
{'mention_rows': 10215104, 'unique_permnos': 3785, 'review_rows': 300}


,permno,ticker,times,del
0,16965,FOR,1551776,1
1,79323,ALL,787951,1
2,14503,OUT,675986,1
3,13447,NOW,570916,1
4,52978,HAS,404020,1
5,85926,SEE,340886,1
6,15293,NEXT,212230,1
7,67467,BIG,205974,1
8,92478,FOR,200059,1
9,75285,MAN,179283,1


請手動修改 del；本 notebook 不套用刪除。
